In [1]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER
import cftime
import datetime
from datetime import date
from matplotlib import pyplot
from matplotlib.cm import ScalarMappable
import matplotlib.colors as mcolors
import matplotlib.ticker as mticker
import numpy
import pandas
import xarray as xr

In [2]:
# Define directories
Diri = '../ExtraTrack_Data/Output_Files_V7/'
Output_Diri = '../RCP_Figs/Analysis_Figs_V7.6.1/'
Large_Scale_Diri = '/glade/campaign/univ/upsu0032/Hyperion_ET/'

In [3]:
# Open file
def Open_File(File):
    DF = pandas.read_csv(File)
    DF = DF.drop("Unnamed: 0", axis=1)
    return (DF)

In [4]:
# Open each file
def Files_Open(Scenario, Diri, Subset):
    Data_DF = Open_File(Diri+Scenario+'_Data_'+Subset+'_Output.csv')
    ET_DF = Open_File(Diri+Scenario+'_ET_'+Subset+'_Output.csv')
    Codes_DF = Open_File(Diri+Scenario+'_Codes_Output.csv')
# Edit time format
    Time_Cols = ["ET Begin Time", "ET Complete Time", "Trop Peak Time", "Peak Time", "Genesis Time", "Final Time"]
    for Col in Time_Cols:
        ET_DF[Col] = pandas.to_datetime(ET_DF[Col], errors="coerce")
    Data_DF["Time(Z)"] = pandas.to_datetime(Data_DF["Time(Z)"], errors="coerce")
    return (Data_DF, ET_DF, Codes_DF)

In [5]:
# Create bins
def Create_Bins(Min, Max, Bin_Width):
    Bins = numpy.arange(Min, Max+Bin_Width, Bin_Width)
    return (Bins)

In [6]:
# Number of years for each climate scenario
Num_Years = numpy.array([90,93,93])

In [7]:
# Open files
Control_Data, Control_ET, Control_Codes = Files_Open("Control", Diri, "SubsetB")
RCP45_Data, RCP45_ET, RCP45_Codes = Files_Open("RCP45", Diri, "SubsetB")
RCP85_Data, RCP85_ET, RCP85_Codes = Files_Open("RCP85", Diri, "SubsetB")

In [10]:
xr.open_dataset(Large_Scale_Diri + 'CHEY.VR28.NATL.REF.CAM5.4CLM5.0.dtime900/CHEY.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.cam.h0.2014-07.nc_regrid.nc')

<xarray.Dataset>
Dimensions:    (time: 1, lat: 181, lon: 361, plev: 13, nbnd: 2)
Coordinates:
  * lat        (lat) float64 -90.0 -89.0 -88.0 -87.0 ... 87.0 88.0 89.0 90.0
  * lon        (lon) float64 -180.0 -179.0 -178.0 -177.0 ... 178.0 179.0 180.0
  * plev       (plev) float64 1e+05 9.25e+04 8.5e+04 ... 1.5e+04 1e+04 5e+03
  * time       (time) datetime64[ns] 2014-08-01
Dimensions without coordinates: nbnd
Data variables: (12/74)
    CLDHGH     (time, lat, lon) float32 ...
    CLDICE     (time, plev, lat, lon) float32 ...
    CLDLIQ     (time, plev, lat, lon) float32 ...
    CLDLOW     (time, lat, lon) float32 ...
    CLDMED     (time, lat, lon) float32 ...
    CLDTOT     (time, lat, lon) float32 ...
    ...         ...
    Z3         (time, plev, lat, lon) float32 ...
    area       (lat, lon) float64 ...
    gw         (lat) float64 ...
    lat_bnds   (lat, nbnd) float64 ...
    lon_bnds   (lon, nbnd) float64 ...
    time_bnds  (time, nbnd) datetime64[ns] ...
Attributes: (12/21)
    ne:                         0
    np:                         4
    Conventions:                CF-1.0
    source:                     CAM
    title:                      Regridded version of tmp.nc
    Version:                    $Name$
    ...                         ...
    history_of_appended_files:  Sun Apr 13 14:45:18 2025: Appended file /glad...
    remap_script:               ncremap
    remap_hostname:             casper-login1
    remap_version:              5.3.1
    map_file:                   /glade/work/zarzycki/maps/hyperion/map_ne0np4...
    input_file:                 /glade/u/home/zarzycki/scratch/hyperion/CHEY....

In [12]:
xr.open_dataset(Large_Scale_Diri + 'CORI.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.003/CORI.VR28.NATL.REF.CAM5.4CLM5.0.dtime900.003.cam.h2.2014-07-28-00000.nc')

<xarray.Dataset>
Dimensions:   (time: 24, lat: 241, lon: 481, lev: 4)
Coordinates:
  * lat       (lat) float64 0.0 0.25 0.5 0.75 1.0 ... 59.0 59.25 59.5 59.75 60.0
  * lev       (lev) float32 300.0 500.0 600.0 900.0
  * lon       (lon) float64 -105.0 -104.8 -104.5 -104.2 ... 14.5 14.75 15.0
  * time      (time) datetime64[ns] 2014-07-28 ... 2014-08-02T18:00:00
Data variables:
    DZ300500  (time, lat, lon) float32 ...
    PS        (time, lat, lon) float32 ...
    PSL       (time, lat, lon) float32 ...
    U850      (time, lat, lon) float32 ...
    UBOT      (time, lat, lon) float32 ...
    V850      (time, lat, lon) float32 ...
    VBOT      (time, lat, lon) float32 ...
    Z         (time, lev, lat, lon) float32 ...
    Z300      (time, lat, lon) float32 ...
    Z500      (time, lat, lon) float32 ...
Attributes:
    creation_date:  Wed Jun 28 08:52:10 MDT 2023
    Conventions:    None
    source_file:    /glade/u/home/zarzycki/scratch/hyperion/CORI.VR28.NATL.RE...
    history:        Wed Jun 28 08:52:11 2023: ncks -4 -L 1 -O CORI.VR28.NATL....
    NCO:            netCDF Operators version 5.1.4 (Homepage = http://nco.sf....